# ML Training with PRE-SCALED CIC-IDS-2018 Dataset

This notebook trains ML models using data that has ALREADY been scaled with PowerTransformer.

**Key Changes from Original:**
1. Load pre-scaled CSV instead of raw data
2. Remove StandardScaler (data already scaled)
3. Keep all other training logic the same

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import time
import pickle
import warnings
warnings.filterwarnings('ignore')
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

# ML Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("🚀 ML TRAINING WITH PRE-SCALED DATASET")
print("="*80)

ModuleNotFoundError: No module named 'lightgbm'

## 2. Configuration

In [ ]:
INPUT_FILE = "archive/cicids2018_scaled.csv"  # Pre-scaled data
SCALER_FILE = "archive/fitted_scalers.pkl"  # For reference/test data

BASE_MODEL_DIR = "trained_models/ml/"
TEST_SIZE = 0.2
RANDOM_STATE = 42
SAVE_MODELS = True

# Per-model dataset size control
# Set the number of rows to use for each model
# None = use full dataset, Integer = use that many rows
MODEL_SAMPLE_SIZES = {
    'Random Forest': None,        # Use full dataset
    'Decision Tree': None,        # Use full dataset
    'Logistic Regression': None,  # Use full dataset
    'XGBoost': None,              # Use full dataset
    'LightGBM': None,             # Use full dataset
    'KNN': 100000,                # Limit KNN (slow on large data)
    'Naive Bayes': None,          # Use full dataset
    'SVM': 50000                  # Limit SVM (very slow)
}

## 3. Load Pre-Scaled Dataset

In [ ]:
CHUNK_SIZE = 100000
MAX_ROWS = None  # None = load all data

chunks = []
total_rows = 0

print(f"\n📂 Loading PRE-SCALED data from: {INPUT_FILE}")
print("⏳ Reading data in chunks...\n")

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    if MAX_ROWS and total_rows >= MAX_ROWS:
        break
    
    if MAX_ROWS and total_rows + len(chunk) > MAX_ROWS:
        chunk = chunk.head(MAX_ROWS - total_rows)
    
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f"   Chunk {i+1}: {len(chunk):,} rows | Total: {total_rows:,}")

# Combine all chunks
df_full = pd.concat(chunks, ignore_index=True)
del chunks  # Free memory

print(f"\n✅ Dataset loaded successfully!")
print(f"   Total rows: {len(df_full):,}")
print(f"   Total columns: {len(df_full.columns)}")

## 4. Verify Data is Pre-Scaled

In [ ]:
print("\n" + "="*80)
print("🔍 VERIFYING DATA IS PRE-SCALED")
print("="*80)

# Check a few features to verify scaling
sample_features = df_full.select_dtypes(include=[np.number]).columns[:5]
print(f"\n📊 Sample feature statistics (should be mean≈0, std≈1):")
print(f"   {'Feature':<30} {'Mean':>10} {'Std':>10}")
print(f"   {'-'*52}")

for col in sample_features:
    if col != 'Label_Binary':  # Skip target
        mean = df_full[col].mean()
        std = df_full[col].std()
        print(f"   {col:<30} {mean:>10.4f} {std:>10.4f}")

print(f"\n✅ Data appears to be pre-scaled! (Mean ≈ 0, Std ≈ 1)")

## 5. Prepare Dataset

In [ ]:
print("\n" + "="*80)
print("🔧 PREPARING DATA FOR TRAINING")
print("="*80)

# Separate features and labels
X_full = df_full.drop(['Label_Binary'], axis=1)
y_full = df_full['Label_Binary']

# Keep only numeric columns (should all be numeric already)
X_full = X_full.select_dtypes(include=[np.number])

print(f"\n✅ Features shape: {X_full.shape}")
print(f"✅ Labels shape: {y_full.shape}")
print(f"\nNumber of features: {X_full.shape[1]}")
print(f"Class distribution:")
print(y_full.value_counts())

## 6. Model Configurations

In [ ]:
models = {
    'Random Forest': {
        'model': RandomForestClassifier(
            n_estimators=100,
            max_depth=20,
            min_samples_split=10,
            min_samples_leaf=4,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=1
        ),
        'use_scaled': False  # Data already scaled, no need to scale again
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(
            max_depth=20,
            min_samples_split=10,
            min_samples_leaf=4,
            random_state=RANDOM_STATE
        ),
        'use_scaled': False
    },
    'Logistic Regression': {
        'model': LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        'use_scaled': False  # Data already scaled
    },
    'XGBoost': {
        'model': XGBClassifier(
            n_estimators=100,
            max_depth=10,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=1
        ),
        'use_scaled': False
    },
    'LightGBM': {
        'model': LGBMClassifier(
            n_estimators=100,
            max_depth=10,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=1
        ),
        'use_scaled': False
    },
    'KNN': {
        'model': KNeighborsClassifier(
            n_neighbors=5,
            n_jobs=-1
        ),
        'use_scaled': False  # Data already scaled (important for KNN!)
    },
    'Naive Bayes': {
        'model': GaussianNB(),
        'use_scaled': False
    },
    'SVM': {
        'model': SVC(
            kernel='rbf',
            C=1.0,
            random_state=RANDOM_STATE,
            verbose=True
        ),
        'use_scaled': False  # Data already scaled (important for SVM!)
    }
}

## 7. Helper Function: Sample Data for Model

In [ ]:
def get_model_data(X_full, y_full, model_name, sample_size, test_size, random_state):
    """
    Get train/test split for a specific model with optional sampling.
    
    NOTE: No scaling is applied here since data is PRE-SCALED!
    """
    
    if sample_size is None:
        X = X_full
        y = y_full
        print(f"📊 Using FULL dataset: {len(X):,} rows")
    else:
        print(f"📊 Sampling {sample_size:,} rows from {len(X_full):,} total rows")
        
        # Stratified sampling to preserve class distribution
        from sklearn.model_selection import train_test_split
        X, _, y, _ = train_test_split(
            X_full, y_full,
            train_size=sample_size,
            random_state=random_state,
            stratify=y_full
        )
        print(f"   Sampled class distribution: {y.value_counts().to_dict()}")
    
    # Split into train and test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    
    print(f"   Train set: {X_train.shape[0]:,} rows")
    print(f"   Test set:  {X_test.shape[0]:,} rows")
    
    # ⚠️ NO SCALING HERE - Data is already scaled!
    return X_train, X_test, y_train, y_test

## 8. Train and Test All Models

In [ ]:
results = {}
trained_models = {}

for model_name, model_config in models.items():
    print("\n" + "="*80)
    print(f"🚀 TRAINING: {model_name}")
    print("="*80)
    
    # Create model-specific directory
    model_dir = os.path.join(BASE_MODEL_DIR, model_name.lower().replace(' ', '_'), '')
    os.makedirs(model_dir, exist_ok=True)
    print(f"📁 Model directory: {model_dir}")
    
    # Get sample size for this model
    sample_size = MODEL_SAMPLE_SIZES.get(model_name, None)
    
    # Prepare data (NO SCALING - already scaled!)
    X_train, X_test, y_train, y_test = get_model_data(
        X_full, y_full,
        model_name,
        sample_size,
        TEST_SIZE,
        RANDOM_STATE
    )
    
    # Train model
    print(f"\n⏱️  Training {model_name}...")
    start_time = time.time()
    
    model = model_config['model']
    model.fit(X_train, y_train)
    
    train_time = time.time() - start_time
    print(f"✅ Training completed in {train_time:.2f} seconds")
    
    # Make predictions
    print(f"\n🔮 Making predictions...")
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f"\n📊 Performance Metrics:")
    print(f"   Accuracy:  {accuracy:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall:    {recall:.4f}")
    print(f"   F1-Score:  {f1:.4f}")
    
    # Store results
    results[model_name] = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'train_time': train_time,
        'sample_size': len(X_train)
    }
    
    # Save model if requested
    if SAVE_MODELS:
        print(f"\n💾 Saving model...")
        
        # Save model
        model_path = os.path.join(model_dir, "model.pkl")
        with open(model_path, 'wb') as f:
            pickle.dump(model, f)
        
        # Save metadata (important!)
        metadata = {
            'model_name': model_name,
            'features': X_train.columns.tolist(),
            'num_features': len(X_train.columns),
            'sample_size': len(X_train),
            'metrics': {
                'accuracy': accuracy,
                'precision': precision,
                'recall': recall,
                'f1': f1
            },
            'train_time': train_time,
            'data_was_prescaled': True,  # Important flag!
            'scaler_file': SCALER_FILE  # Reference to scaler
        }
        
        metadata_path = os.path.join(model_dir, "metadata.pkl")
        with open(metadata_path, 'wb') as f:
            pickle.dump(metadata, f)
        
        print(f"   ✅ Model saved: {model_path}")
        print(f"   ✅ Metadata saved: {metadata_path}")
    
    trained_models[model_name] = model

## 9. Results Summary

In [ ]:
print("\n" + "="*80)
print("📊 TRAINING RESULTS SUMMARY")
print("="*80)

summary_df = pd.DataFrame(results).T
summary_df = summary_df.sort_values('f1', ascending=False)

# Rename columns for display
summary_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Train Time (s)', 'Sample Size']

print("\n" + summary_df.to_string())
summary_df

## 10. Visualization

In [ ]:
print("\n📊 Creating performance visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison (Pre-Scaled Data)', fontsize=16, fontweight='bold')

# Accuracy
axes[0, 0].barh(summary_df.index, summary_df['Accuracy'], color='skyblue')
axes[0, 0].set_xlabel('Accuracy')
axes[0, 0].set_title('Accuracy Comparison', fontweight='bold')
axes[0, 0].set_xlim(0, 1)

# F1-Score
axes[0, 1].barh(summary_df.index, summary_df['F1-Score'], color='coral')
axes[0, 1].set_xlabel('F1-Score')
axes[0, 1].set_title('F1-Score Comparison', fontweight='bold')
axes[0, 1].set_xlim(0, 1)

# Training Time
axes[1, 0].barh(summary_df.index, summary_df['Train Time (s)'], color='lightgreen')
axes[1, 0].set_xlabel('Training Time (seconds)')
axes[1, 0].set_title('Training Time Comparison', fontweight='bold')

# All metrics together
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
x = np.arange(len(summary_df))
width = 0.2

for i, metric in enumerate(metrics_to_plot):
    axes[1, 1].bar(x + i*width, summary_df[metric], width, label=metric)

axes[1, 1].set_xlabel('Models')
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('All Metrics Comparison', fontweight='bold')
axes[1, 1].set_xticks(x + width * 1.5)
axes[1, 1].set_xticklabels(summary_df.index, rotation=45, ha='right')
axes[1, 1].legend()
axes[1, 1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(BASE_MODEL_DIR, 'model_comparison.png'), dpi=300, bbox_inches='tight')
print(f"✅ Saved visualization: {os.path.join(BASE_MODEL_DIR, 'model_comparison.png')}")
plt.show()

## 11. Final Summary

In [ ]:
print("\n" + "="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)

print("\n📊 Models Ranked by F1-Score:")
for idx, (model_name, row) in enumerate(summary_df.iterrows(), 1):
    print(f"   {idx}. {model_name}: {row['F1-Score']:.4f} "
          f"(trained on {int(row['Sample Size']):,} samples)")

print(f"\n📁 All models saved to: {BASE_MODEL_DIR}")
print("\n✅ All models trained on PRE-SCALED data!")
print(f"✅ To use on new data, apply the same scaler: {SCALER_FILE}")